# data_cleaning_23301

In [1]:
import pandas as pd
import numpy as np

# 1. 读取数据

In [2]:
file_path = './Data/Online Retail.xlsx'
df = pd.read_excel(file_path)

print(f"原始数据形状: {df.shape}")

原始数据形状: (541909, 8)


# 2. 清洗列名

In [3]:
df.columns = (
    df.columns
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .str.replace(r'[\u200b\u200c\u200d\ufeff]', '', regex=True)  # 零宽字符
)

print("=== 清洗后的列名 ===")
print(df.columns.tolist())

=== 清洗后的列名 ===
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


# 3. 列名存在性检查

In [4]:
required_cols = ['InvoiceNo', 'StockCode', 'Description', 'Quantity',
                 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']
missing = [c for c in required_cols if c not in df.columns]
if missing:
    print(f"警告: 缺少以下列: {missing}")
else:
    print("所有核心列均已找到！")

所有核心列均已找到！


# 4. 数据类型转换

In [5]:
# ============================
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')
df['InvoiceNo'] = df['InvoiceNo'].astype(str)
df['StockCode'] = df['StockCode'].astype(str)

# CustomerID 转为字符串（处理浮点数形式）
df['CustomerID'] = df['CustomerID'].astype(str).str.replace('.0', '', regex=False)
df['CustomerID'] = df['CustomerID'].replace({'nan': np.nan, 'None': np.nan, '': np.nan})

# 5. 筛选目标商品 StockCode = 23301

In [6]:
TARGET_CODE = '23301'
before_filter = len(df)
df = df[df['StockCode'] == TARGET_CODE].copy()
print(f"筛选 StockCode = {TARGET_CODE}: {before_filter} → {len(df)} 行")

if df.empty:
    print(f"错误: 未找到 StockCode = {TARGET_CODE} 的记录，请确认编码是否正确。")
    exit()

筛选 StockCode = 23301: 541909 → 937 行


# 6. 缺失值处理

In [7]:
# Description 缺失：用相同 StockCode 的众数填充，否则标记 Unknown
desc_map = df.groupby('StockCode')['Description'].agg(
    lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
)
df['Description'] = df['Description'].fillna(df['StockCode'].map(desc_map))
df['Description'] = df['Description'].fillna('Unknown')

# CustomerID 缺失：删除
before = len(df)
df = df.dropna(subset=['CustomerID'])
print(f"删除 CustomerID 缺失行: {before - len(df)} 行")

print("=== 清洗后缺失值统计 ===")
print(df.isnull().sum())

删除 CustomerID 缺失行: 158 行
=== 清洗后缺失值统计 ===
InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64


# 7. 异常值与业务逻辑清洗

In [8]:
# 分离退货单（InvoiceNo 以 C 开头）
mask_cancelled = df['InvoiceNo'].str.startswith('C', na=False)
df_cancel = df[mask_cancelled].copy()
df_normal = df[~mask_cancelled].copy()

# 正常订单 Quantity 必须为正数
df_normal = df_normal[df_normal['Quantity'] > 0]

# 合并（如不需要退货数据，可只保留 df_normal）
df = pd.concat([df_normal, df_cancel], ignore_index=True)

# UnitPrice 必须为正数
df = df[df['UnitPrice'] > 0]

# 8. 重复值处理

In [9]:
before_dup = len(df)
df = df.drop_duplicates(keep='first')
print(f"删除重复行: {before_dup - len(df)} 行")

删除重复行: 6 行


# 9. 衍生特征

In [10]:
# 计算总金额
df['TotalAmount'] = df['Quantity'] * df['UnitPrice']

# 提取时间特征
df['Year'] = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month
df['Day'] = df['InvoiceDate'].dt.day
df['Hour'] = df['InvoiceDate'].dt.hour
df['DayOfWeek'] = df['InvoiceDate'].dt.dayofweek  # 0=周一, 6=周日

# 10. Country 标准化

In [11]:
df['Country'] = df['Country'].str.strip().str.title()

# 11. 保存结果

In [12]:
print(f"最终数据形状: {df.shape}")
print("前5行预览:")
print(df.head())

output_path = './Data/Online_Retail_23301_Cleaned.xlsx'
df.to_excel(output_path, index=False)
print(f"清洗后的数据已保存至: {output_path}")

最终数据形状: (773, 14)
前5行预览:
  InvoiceNo StockCode                        Description  Quantity  \
0    553181     23301  GARDENERS KNEELING PAD KEEP CALM         12   
1    553182     23301  GARDENERS KNEELING PAD KEEP CALM          6   
2    553184     23301  GARDENERS KNEELING PAD KEEP CALM          2   
3    553189     23301  GARDENERS KNEELING PAD KEEP CALM          2   
4    553190     23301  GARDENERS KNEELING PAD KEEP CALM          1   

          InvoiceDate  UnitPrice CustomerID         Country  TotalAmount  \
0 2011-05-15 11:43:00       1.65      16168  United Kingdom        19.80   
1 2011-05-15 12:21:00       1.65      17107  United Kingdom         9.90   
2 2011-05-15 12:33:00       1.65      13232  United Kingdom         3.30   
3 2011-05-15 13:14:00       1.65      15033  United Kingdom         3.30   
4 2011-05-15 13:14:00       1.65      15164  United Kingdom         1.65   

   Year  Month  Day  Hour  DayOfWeek  
0  2011      5   15    11          6  
1  2011      5   15